# Stratified profile bootstrap

Run from the project root or `analysis/`. Each of 1,000 resamples draws 14 profiles within each of the 16 Gender × Race × injection-experience strata, retaining both housing rows. Models and prompt versions use the same draws.

Results cover 32 relationships under the original (0) and paraphrased (2) prompts. This notebook reports directional stability and estimability without confidence intervals.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import time

ROOT = next((candidate for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent)
             if (candidate / "analysis" / "analysis_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the project root or its analysis folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
from analysis.analysis_utils import (DATA_DIR, RESULT_DIR, MODELS, TARGETS, COLUMNS,
    PERCENTAGE_POINT_OUTCOMES, HOUSING_UNITS, rank_correlation,
    STRATA_COLUMNS, reconstruct_original_profiles, stratified_profile_draws)

B = 1000
SEED = 42
PROMPT_VERSIONS = [0, 2]
EXPECTED_PROFILES = 224
RAW_HASHES_BEFORE = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                     for path in DATA_DIR.iterdir() if path.suffix in {".csv", ".jsonl"}}


## Profile strata

Strata use unrounded CNEP+ ages and the 224 saved profile IDs. `Experience_group` is `Young` for less than 3 years of injecting and `Old` for at least 3 years.


In [ ]:
outputs = pd.read_csv(DATA_DIR / "outputs.csv")
outputs = outputs.loc[outputs.Paraphrase.isin(PROMPT_VERSIONS)].copy()
profiles = reconstruct_original_profiles(ROOT / "2018-09-13--06.02.24.events.csv", set(outputs.Agent))
stratum_counts = profiles.groupby(STRATA_COLUMNS, sort=True).size().rename("N_profiles").reset_index()
print(stratum_counts.to_string(index=False))
metadata = outputs[["Agent", "Age", "Age_Started"]].drop_duplicates()
if metadata.Agent.duplicated().any():
    raise ValueError("Profile age metadata differs across saved rows.")
metadata = profiles.merge(metadata, on="Agent", validate="one_to_one")
rounded_age_stratum_changes = int(((metadata.Age-metadata.Age_Started).lt(3)
                                 != metadata.Injection_Experience.lt(3)).sum())
print(f"\nOriginal cohort recovered exactly. Rounded CSV ages would misclassify {rounded_age_stratum_changes} profiles.")


## Estimates and directions

- Housing relationships (9): mean homeless-minus-stable difference from complete profile pairs; proportions and binary outcomes use percentage points
- Correlations (23): pooled-housing Spearman correlation, requiring at least three complete observations and variation in both variables
- Directions use unrounded estimates; zero does not match an expected increase or decrease
- The four mixed-expectation relationships have no directional agreement score


In [ ]:
def prepare_relationships():
    definitions = []
    for number, target in enumerate(TARGETS, 1):
        x, y = COLUMNS[target["exposure"]], COLUMNS[target["outcome"]]
        kind = "housing" if x == "Homeless" else "correlation"
        definitions.append(dict(Relationship_id=number, Kind=kind, Exposure=target["exposure"],
            Outcome=target["outcome"], X=x, Y=y, Expected_direction=target["expected"],
            Fixed_direction=target["expected"] in ["↑", "↓"],
            Scope="homeless minus stable" if kind == "housing" else "pooled housing conditions",
            Unit=("percentage points" if y in PERCENTAGE_POINT_OUTCOMES else HOUSING_UNITS[y]) if kind == "housing" else "Spearman rho"))
    return definitions

definitions = prepare_relationships()
MEASURES = list(dict.fromkeys(column for definition in definitions for column in [definition["X"], definition["Y"]] if column != "Homeless"))
MEASURE_INDEX = {name: index for index, name in enumerate(MEASURES)}
assert len(definitions) == 32 and sum(item["Fixed_direction"] for item in definitions) == 28
assert sum(item["Kind"] == "housing" for item in definitions) == 9

def prepare_output_arrays(frame, profile_ids, prompt_versions):
    frame = frame.copy()
    keys = ["Model", "Paraphrase", "Agent", "Homeless"]
    if frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError("Duplicate or missing saved-output identifiers.")
    if set(frame.Model) != set(MODELS) or not frame.Homeless.isin([0, 1]).all():
        raise ValueError("Unexpected model or housing labels.")
    hcv = frame.HCV.astype("string").str.strip().str.lower()
    if not hcv.dropna().isin(["susceptible", "acute"]).all():
        raise ValueError("Unrecognized HCV state.")
    frame["HCV_active"] = hcv.map({"susceptible": 0, "acute": 1}).astype(float)
    for variable in MEASURES:
        frame[variable] = pd.to_numeric(frame[variable], errors="raise")
        if np.isinf(frame[variable]).any() or frame[variable].dropna().lt(0).any():
            raise ValueError(f"Invalid numeric output: {variable}")
    for variable in PERCENTAGE_POINT_OUTCOMES:
        if frame[variable].dropna().gt(1).any():
            raise ValueError(f"Out-of-range proportion/binary output: {variable}")
    if not frame.Treatment.dropna().isin([0, 1]).all():
        raise ValueError("Treatment must be binary.")
    expected_index = pd.MultiIndex.from_product([profile_ids, [0, 1]], names=["Agent", "Homeless"])
    arrays = {}
    for model in MODELS:
        for version in prompt_versions:
            group = frame.loc[frame.Model.eq(model) & frame.Paraphrase.eq(version)]
            if set(zip(group.Agent, group.Homeless)) != set(expected_index):
                raise ValueError(f"{model}/{version}: expected both housing outputs for every profile.")
            ordered = group.set_index(["Agent", "Homeless"]).reindex(expected_index)
            arrays[(model, version)] = ordered[MEASURES].to_numpy(float).reshape(len(profile_ids), 2, len(MEASURES))
    return arrays

arrays = prepare_output_arrays(outputs, profiles.Agent.to_numpy(), PROMPT_VERSIONS)
print(f"Prepared {len(arrays)} model/prompt datasets, each with {len(profiles)} paired profiles.")


In [ ]:
draws = stratified_profile_draws(profiles, B, SEED)
unique_profiles_per_draw = np.array([len(np.unique(indices)) for indices in draws])
DRAW_INDEX_SHA256 = hashlib.sha256(draws.astype("<i8").tobytes()).hexdigest()
print(f"Created {B:,} common stratified draws of {draws.shape[1]} profile slots, retaining both housing rows per slot.")
print(f"Distinct profiles per draw: min={unique_profiles_per_draw.min()}, max={unique_profiles_per_draw.max()}.")
print(f"Draw-index fingerprint: {DRAW_INDEX_SHA256}")


In [ ]:
def evaluate_relationship(values, definition):
    """Point estimate only: no nested bootstrap or confidence interval."""
    invariant_x = invariant_y = None
    reason = ""
    if definition["Kind"] == "housing":
        pair = values[:, :, MEASURE_INDEX[definition["Y"]]]
        complete = pair[np.isfinite(pair).all(axis=1)]
        n_complete = len(complete)
        scale = 100 if definition["Y"] in PERCENTAGE_POINT_OUTCOMES else 1
        estimate = float((complete[:, 1]-complete[:, 0]).mean()) * scale if n_complete else np.nan
        if not n_complete:
            reason = "no_complete_profile_pairs"
    else:
        i, j = MEASURE_INDEX[definition["X"]], MEASURE_INDEX[definition["Y"]]
        pair = values[:, :, [i, j]].reshape(-1, 2)
        complete = pair[np.isfinite(pair).all(axis=1)]
        n_complete = len(complete)
        invariant_x = bool(n_complete and np.ptp(complete[:, 0]) == 0)
        invariant_y = bool(n_complete and np.ptp(complete[:, 1]) == 0)
        estimate = np.nan
        if n_complete < 3:
            reason = "fewer_than_three_complete_observations"
        elif invariant_x or invariant_y:
            reason = "both_variables_invariant" if invariant_x and invariant_y else "x_invariant" if invariant_x else "y_invariant"
        else:
            estimate = float(rank_correlation(complete[:, 0], complete[:, 1]))
            if not np.isfinite(estimate):
                reason = "correlation_undefined"
    estimable = bool(np.isfinite(estimate))
    direction = ("positive" if estimate > 0 else "negative" if estimate < 0 else "zero") if estimable else None
    agreement = None
    if estimable and definition["Fixed_direction"]:
        expected_sign = 1 if definition["Expected_direction"] == "↑" else -1
        agreement = bool(np.sign(estimate) == expected_sign)
    return estimate, estimable, direction, agreement, invariant_x, invariant_y, n_complete, reason

def calculate_bootstrap(arrays_by_context, definitions, draw_indices):
    observed_rows, records = [], []
    started = time.monotonic()
    for (model, version), values in arrays_by_context.items():
        for definition in definitions:
            estimate, estimable, direction, agreement, invariant_x, invariant_y, n_complete, reason = evaluate_relationship(values, definition)
            observed_rows.append(dict(Model=model, Paraphrase=version, **definition,
                Observed_estimate=estimate, Observed_estimable=estimable, Observed_direction=direction,
                Observed_agreement=agreement, Observed_invariant_x=invariant_x, Observed_invariant_y=invariant_y,
                Observed_n_complete=n_complete, Observed_nonestimable_reason=reason))
        for replicate, indices in enumerate(draw_indices, 1):
            # Keep repeated profile draws
            selected = values[indices, :, :]
            for definition in definitions:
                result = evaluate_relationship(selected, definition)
                records.append((model, version, replicate, definition["Relationship_id"], *result))
        print(f"Completed {model}, Paraphrase {version}: {len(draw_indices):,} stratified paired replicates ({time.monotonic()-started:.1f}s elapsed).", flush=True)
    columns = ["Model", "Paraphrase", "Resample", "Relationship_id", "Estimate", "Estimable", "Direction",
               "Directional_agreement", "Invariant_x", "Invariant_y", "N_complete", "Nonestimable_reason"]
    records = pd.DataFrame.from_records(records, columns=columns)
    for column in ["Directional_agreement", "Invariant_x", "Invariant_y"]:
        records[column] = pd.array(records[column], dtype="boolean")
    observed = pd.DataFrame(observed_rows)
    observed["Observed_agreement"] = pd.array(observed.Observed_agreement, dtype="boolean")
    return observed, records

observed, bootstrap_records = calculate_bootstrap(arrays, definitions, draws)
assert len(bootstrap_records) == len(arrays) * len(definitions) * B
print(f"\nRecorded {len(bootstrap_records):,} model/prompt/relationship/resample estimates.")


## Bootstrap summaries

`Directional_stability_pct = 100 × N_agree / N_direction_assessed`, where `N_direction_assessed` counts estimable resamples. Stability is missing for mixed expectations or zero estimable resamples.

`Estimable_pct = 100 × N_estimable / 1000`. Non-estimable resamples reduce estimability and are excluded from directional agreement.


In [ ]:
def summarize_bootstrap(observed_frame, records, n_resamples):
    rows = []
    keys = ["Model", "Paraphrase", "Relationship_id"]
    observed_lookup = observed_frame.set_index(keys)
    for key, group in records.groupby(keys, sort=True):
        original = observed_lookup.loc[key]
        if len(group) != n_resamples or group.Resample.nunique() != n_resamples:
            raise ValueError("Missing or duplicate bootstrap replicates.")
        estimable = group.Estimable.astype(bool)
        n_estimable = int(estimable.sum())
        fixed = bool(original.Fixed_direction)
        if fixed:
            assert group.loc[estimable, "Directional_agreement"].notna().all()
            n_agree = int(group.loc[estimable, "Directional_agreement"].sum())
            n_disagree = n_estimable - n_agree
            denominator = n_estimable
            stability = 100*n_agree/n_estimable if n_estimable else np.nan
        else:
            n_agree = n_disagree = pd.NA
            denominator = 0
            stability = np.nan
        assert group.loc[~estimable, "Directional_agreement"].isna().all()
        if not fixed:
            assert group.Directional_agreement.isna().all()
        rows.append(original.to_dict() | dict(zip(keys, key)) | dict(
            B=n_resamples, N_estimable=n_estimable, N_nonestimable=n_resamples-n_estimable,
            Estimable_pct=100*n_estimable/n_resamples, N_direction_assessed=denominator,
            N_agree=n_agree, N_disagree=n_disagree, Directional_stability_pct=stability,
            N_zero_estimates=int((estimable & group.Estimate.eq(0)).sum())))
    result = pd.DataFrame(rows)
    result["Observed_agreement"] = pd.array(result.Observed_agreement, dtype="boolean")
    for column in ["N_agree", "N_disagree"]:
        result[column] = pd.array(result[column], dtype="Int64")
    return result

bootstrap_summary = summarize_bootstrap(observed, bootstrap_records, B)
assert len(bootstrap_summary) == len(arrays) * len(definitions)
constant_original = bootstrap_summary.Kind.eq("correlation") & (
    bootstrap_summary.Observed_invariant_x.fillna(False).astype(bool)
    | bootstrap_summary.Observed_invariant_y.fillna(False).astype(bool))
assert bootstrap_summary.loc[constant_original, "N_estimable"].eq(0).all()
print("Original prompt: observed agreement, bootstrap directional stability, and estimability")
columns = ["Model", "Relationship_id", "Kind", "X", "Y", "Expected_direction", "Observed_direction",
           "Observed_agreement", "N_estimable", "B", "Estimable_pct", "N_agree", "N_direction_assessed", "Directional_stability_pct"]
print(bootstrap_summary.loc[bootstrap_summary.Paraphrase.eq(0), columns].to_string(index=False))
print("\nRelationships that were estimable in the observed sample but lost estimability in some resamples")
fragile = bootstrap_summary.loc[bootstrap_summary.Observed_estimable & bootstrap_summary.N_estimable.lt(B)]
print(fragile[["Model", "Paraphrase", "Relationship_id", "X", "Y", "N_estimable", "B", "Estimable_pct", "Directional_stability_pct"]].to_string(index=False))


## Save results

- `bootstrap_resamples.csv`: estimates and estimability/agreement flags for each resample
- `bootstrap_summary.csv`: observed estimates, bootstrap percentages, and their counts

Housing `N_complete` counts profile pairs; correlation `N_complete` counts pooled housing rows


In [ ]:
RESULT_DIR.mkdir(exist_ok=True)
bootstrap_records.to_csv(RESULT_DIR / "bootstrap_resamples.csv", index=False)
bootstrap_summary.to_csv(RESULT_DIR / "bootstrap_summary.csv", index=False)
RAW_HASHES_AFTER = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                    for path in DATA_DIR.iterdir() if path.suffix in {".csv", ".jsonl"}}
assert RAW_HASHES_BEFORE == RAW_HASHES_AFTER, "Raw CSV/JSONL files changed."
print(f"Saved {len(bootstrap_records):,} replicate records and {len(bootstrap_summary)} relationship summaries.")
print(f"All {len(RAW_HASHES_AFTER)} raw CSV/JSONL files are unchanged. No confidence intervals were calculated.")
